# Outlier detection

This notebook decides how unusual values should be handled in this project. The decision is based on data meaning and temporal cross-validation, not only on how many rows an algorithm flags.

## Principles

- An impossible value is a data error and is handled by a consistency rule.
- An extreme value is not automatically an error. Large purchases and long customer histories may be valid.
- The most recent 20% of the training period remains locked and is not used to choose a method.
- Transformations and outlier detectors are fitted on each training fold only. Validation rows are never removed.
- Rows from `test.csv` are never removed.

## 0. Imports and configuration

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from minisom import MiniSom
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import DBSCAN, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

import classification.preprocessing as classification_prep
import regression.preprocessing as regression_prep
from classification.model import build_model as build_classifier
from regression.model import build_model as build_regressor
from src.evaluate import (
    cross_validate_pipeline, make_pipeline as build_model_pipeline,
    paired_comparison, temporal_folds,
)
from src.preprocessing import (
    build_preprocessor, drop_training_rows, load_data, prepare_table,
    split_dev_test, standardize_columns, task_preprocessing_config,
)

warnings.filterwarnings("ignore")

with open("config.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_cfg = config["data"]
diag_cfg = config["diagnostics"]

## 1. Load the cleaned development data

The same project functions used by `main.py` apply the cleaning rules. The 65 almost-empty rows are removed. The recent locked period is set aside before any outlier decision is made.

In [ ]:
train_raw = load_data(data_cfg["train_path"])
test_raw = load_data(data_cfg["test_path"])
clientes_raw = load_data(data_cfg["clientes_path"])

train_clean, cleaning_report = prepare_table(
    train_raw, config, clientes_raw
)
train_clean, row_report = drop_training_rows(
    train_clean, diag_cfg, data_cfg["id_column"], config["customers"]
)

classification_rows = classification_prep.select_training_rows(
    train_clean, config
)
X, y, extras = classification_prep.prepare_features(
    classification_rows, config
)
X_dev, X_locked, y_dev, y_locked, extras_dev, extras_locked = (
    split_dev_test(
        X, y, extras, config["test_set"], data_cfg["date_column"]
    )
)

print("Raw training rows:", len(train_raw))
print("Rows after cleaning:", len(train_clean))
print("Development rows analysed:", len(X_dev))
print("Locked rows not analysed:", len(X_locked))

## 2. Separate data errors from valid extremes

A previous maximum purchase cannot be larger than the sum of all previous purchases. This is a logical error, not a statistical outlier. The rule is now in `config.yaml`; invalid values become missing and the fitted median imputer handles them inside each fold.

In [ ]:
def count_impossible_prior_max(raw_data):
    data = standardize_columns(raw_data, diag_cfg["column_renames"])
    is_error = (
        data["prior_purchase_count"].gt(0)
        & data["prior_max_purchase_value"].notna()
        & data["prior_total_spend"].notna()
        & data["prior_max_purchase_value"].gt(
            data["prior_total_spend"] + 0.01
        )
    )
    return int(is_error.sum())

print("Impossible values in train:", count_impossible_prior_max(train_raw))
print("Impossible values in test: ", count_impossible_prior_max(test_raw))

## 3. Numerical overview

Targets, identifiers and dates are excluded. Only numerical predictors used by the classification recipe are inspected here.

In [ ]:
classification_cfg = task_preprocessing_config(config, "classification")
numeric_columns = [
    column for column in classification_cfg["numeric_features"]
    if column in X_dev.columns
]
numeric_data = X_dev[numeric_columns].copy()

print("Numerical columns analysed:", len(numeric_columns))
display(numeric_data.describe().T)

## 4. IQR and skewness report

The IQR rule is a useful warning signal, not a deletion rule. It is not informative when the IQR is zero, which happens in sparse or almost-constant columns.

In [ ]:
report_rows = []

for column in numeric_columns:
    values = numeric_data[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        outlier_count = np.nan
        outlier_percent = np.nan
        status = "Not informative (IQR = 0)"
    else:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        is_outlier = values.lt(lower) | values.gt(upper)
        outlier_count = int(is_outlier.sum())
        outlier_percent = is_outlier.mean() * 100
        status = "Available"

    report_rows.append({
        "column": column,
        "missing_%": numeric_data[column].isna().mean() * 100,
        "zero_%": values.eq(0).mean() * 100,
        "skewness": values.skew(),
        "iqr_status": status,
        "iqr_outliers": outlier_count,
        "iqr_outlier_%": outlier_percent,
    })

outlier_report = pd.DataFrame(report_rows).set_index("column")
outlier_report["absolute_skewness"] = outlier_report["skewness"].abs()
display(outlier_report.sort_values("absolute_skewness", ascending=False).round(2))

## 5. Inspect the strongest values

The six most skewed predictors are plotted, then their largest values are shown with row identifiers. The identifiers are for investigation only and are not model inputs.

In [ ]:
top_columns = (
    outlier_report["absolute_skewness"]
    .sort_values(ascending=False)
    .head(6)
    .index
)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for column, axis in zip(top_columns, axes.flat):
    axis.hist(numeric_data[column].dropna(), bins=40)
    axis.set_title(
        f"{column}\nskewness = {outlier_report.loc[column, 'skewness']:.2f}"
    )
plt.tight_layout()
plt.show()

for column in top_columns:
    largest = extras_dev[["id", "customer_id", "purchase_date"]].copy()
    largest[column] = numeric_data[column]
    print(f"\nLargest values of {column}")
    display(largest.nlargest(5, column))

## 6. What do the algorithms actually flag?

IQR, Isolation Forest and LOF are compared on the same development rows. A 3% rate is used for the multivariate methods only to make their groups comparable. The target summaries reveal whether a method mainly selects useful high-value customers.

In [ ]:
def method_flags(features, numeric_features):
    available = [column for column in numeric_features if column in features.columns]
    iqr_count = pd.Series(0, index=features.index)

    for column in available:
        values = features[column]
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            flag = values.lt(q1 - 1.5 * iqr) | values.gt(q3 + 1.5 * iqr)
            iqr_count += flag.fillna(False).astype(int)

    scaler = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    matrix = scaler.fit_transform(features[available])

    forest = IsolationForest(contamination=0.03, random_state=42)
    lof = LocalOutlierFactor(contamination=0.03)

    return pd.DataFrame({
        "IQR (3+ columns)": iqr_count.ge(3),
        "Isolation Forest": forest.fit_predict(matrix) == -1,
        "LOF": lof.fit_predict(matrix) == -1,
    }, index=features.index)

classification_flags = method_flags(X_dev, numeric_columns)

print("Rows flagged by each method")
display(classification_flags.mean().mul(100).round(2).rename("flagged_%"))
print("Pairwise overlap in rows")
display(classification_flags.astype(int).T.dot(classification_flags.astype(int)))

target_profiles = []
for method in classification_flags:
    flag = classification_flags[method]
    target_profiles.append({
        "method": method,
        "repeat rate — flagged": y_dev[flag].mean(),
        "repeat rate — other rows": y_dev[~flag].mean(),
    })
display(pd.DataFrame(target_profiles).set_index("method").round(3))

## 7. Temporal validation setup

Both tasks use five expanding-window folds with the same 90-day gap as the project pipeline. This estimates performance on future purchases and keeps fitted preprocessing out of validation data.

In [ ]:
def prepare_task(task, module, model_builder):
    rows = module.select_training_rows(train_clean, config)
    features, target, row_info = module.prepare_features(rows, config)
    features, _, target, _, row_info, _ = split_dev_test(
        features, target, row_info, config["test_set"], data_cfg["date_column"]
    )

    features = features.reset_index(drop=True)
    target = target.reset_index(drop=True)
    row_info = row_info.reset_index(drop=True)
    folds = temporal_folds(
        row_info[data_cfg["date_column"]],
        config["validation"]["n_splits"],
        config["validation"]["gap_days"],
    )

    return {
        "task": task,
        "X": features,
        "y": target,
        "folds": folds,
        "preprocessing": task_preprocessing_config(config, task),
        "model_builder": model_builder,
        "metric": config[task]["metric"],
    }

task_data = {
    "classification": prepare_task(
        "classification", classification_prep, build_classifier
    ),
    "regression": prepare_task(
        "regression", regression_prep, build_regressor
    ),
}

for task, data in task_data.items():
    print(task, "development rows:", len(data["X"]))

## 8. Compare transformations

`log1p` is fitted only to non-negative columns whose training-fold skewness exceeds 2. Winsorisation learns its limits from each training fold. These are compared with no change and robust scaling.

In [ ]:
class SkewLog1p(BaseEstimator, TransformerMixin):
    def __init__(self, columns, minimum_skew=2):
        self.columns = columns
        self.minimum_skew = minimum_skew

    def fit(self, X, y=None):
        self.columns_ = [
            column for column in self.columns
            if column in X and X[column].min() >= 0
            and abs(X[column].skew()) > self.minimum_skew
        ]
        return self

    def transform(self, X):
        result = X.copy()
        result[self.columns_] = np.log1p(result[self.columns_])
        return result


class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, columns, lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.columns_ = [column for column in self.columns if column in X]
        self.lower_limits_ = X[self.columns_].quantile(self.lower)
        self.upper_limits_ = X[self.columns_].quantile(self.upper)
        return self

    def transform(self, X):
        result = X.copy()
        for column in self.columns_:
            result[column] = result[column].astype("float64").clip(
                self.lower_limits_[column], self.upper_limits_[column]
            )
        return result

In [ ]:
def transformation_results(data):
    task = data["task"]
    prep_cfg = data["preprocessing"]
    numeric = prep_cfg["numeric_features"]
    recipes = {
        "No change": None,
        "log1p for skewed columns": SkewLog1p(numeric),
        "Winsorize 1%–99%": Winsorizer(numeric, 0.01, 0.99),
        "Winsorize 0.5%–99.5%": Winsorizer(numeric, 0.005, 0.995),
    }
    rows = []
    per_fold = {}

    for name, transformer in recipes.items():
        model = data["model_builder"](config[task]["model"])
        steps = [] if transformer is None else [("outlier_transform", transformer)]
        pipeline = Pipeline(steps + [
            ("prep", build_preprocessor(prep_cfg)),
            ("model", model),
        ])
        scores = cross_validate_pipeline(
            pipeline, data["X"], data["y"], data["folds"], task
        )
        values = scores[f"validation_{data['metric']}"].to_numpy()
        per_fold[name] = values
        rows.append({
            "task": task, "method": name,
            "mean_score": values.mean(),
            "fold_std": values.std(ddof=1),
        })

    robust_cfg = {**prep_cfg, "scaler": "robust"}
    robust_scores = cross_validate_pipeline(
        build_model_pipeline(
            robust_cfg, data["model_builder"](config[task]["model"])
        ),
        data["X"], data["y"], data["folds"], task,
    )
    robust_values = robust_scores[f"validation_{data['metric']}"].to_numpy()
    per_fold["Robust scaler"] = robust_values
    rows.append({
        "task": task, "method": "Robust scaler",
        "mean_score": robust_values.mean(),
        "fold_std": robust_values.std(ddof=1),
    })

    baseline = per_fold["No change"]
    for row in rows:
        comparison = paired_comparison(
            per_fold[row["method"]], baseline, task == "classification"
        )
        row["mean_advantage"] = comparison["mean_advantage"]
        row["standard_error"] = comparison["se"]
        row["reliably_better"] = (
            comparison["mean_advantage"] > 2 * comparison["se"]
        )
    return rows

transformation_table = pd.DataFrame(
    transformation_results(task_data["classification"])
    + transformation_results(task_data["regression"])
)
display(transformation_table.round(4))

## 9. Compare row-removal methods

Each detector sees only a training fold. It may remove rows from that fold, but the following validation period is left complete. A method is only accepted if its average advantage is larger than twice its standard error.

In [ ]:
def scaled_numeric_matrix(features, columns):
    available = [column for column in columns if column in features.columns]
    scaler = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    return scaler.fit_transform(features[available])


def rows_to_remove(features, numeric, method):
    if method == "No removal":
        return np.zeros(len(features), dtype=bool)

    if method == "IQR (3+ columns)":
        return method_flags(features, numeric)[method].to_numpy()

    matrix = scaled_numeric_matrix(features, numeric)

    if method == "Isolation Forest":
        return IsolationForest(
            contamination=0.03, random_state=42
        ).fit_predict(matrix) == -1

    if method == "LOF":
        return LocalOutlierFactor(contamination=0.03).fit_predict(matrix) == -1

    if method == "KNN distance":
        distances = NearestNeighbors(n_neighbors=11).fit(matrix).kneighbors(matrix)[0]
        score = distances[:, 1:].mean(axis=1)
        return score >= np.quantile(score, 0.97)

    if method == "K-Means distance":
        kmeans = KMeans(n_clusters=10, n_init=10, random_state=42).fit(matrix)
        score = np.linalg.norm(
            matrix - kmeans.cluster_centers_[kmeans.labels_], axis=1
        )
        return score >= np.quantile(score, 0.97)

    if method == "DBSCAN":
        distances = NearestNeighbors(n_neighbors=10).fit(matrix).kneighbors(matrix)[0]
        epsilon = np.quantile(distances[:, -1], 0.97)
        return DBSCAN(eps=epsilon, min_samples=10).fit_predict(matrix) == -1

    if method == "SOM distance":
        som = MiniSom(
            10, 10, matrix.shape[1], sigma=1, learning_rate=0.5, random_seed=42
        )
        som.random_weights_init(matrix)
        som.train_random(matrix, 1000)
        weights = som.get_weights()
        score = np.array([
            np.linalg.norm(row - weights[som.winner(row)]) for row in matrix
        ])
        return score >= np.quantile(score, 0.97)

    raise ValueError(f"Unknown method: {method}")

In [ ]:
def evaluate_removal(data, method):
    task = data["task"]
    numeric = data["preprocessing"]["numeric_features"]
    scores = []
    removed = []

    for train_positions, validation_positions in data["folds"]:
        X_train = data["X"].iloc[train_positions]
        y_train = data["y"].iloc[train_positions]
        remove = rows_to_remove(X_train, numeric, method)
        removed.append(remove.mean() * 100)

        pipeline = build_model_pipeline(
            data["preprocessing"],
            data["model_builder"](config[task]["model"]),
        )
        pipeline.fit(X_train.loc[~remove], y_train.loc[~remove])
        predictions = pipeline.predict(data["X"].iloc[validation_positions])

        if task == "classification":
            score = accuracy_score(data["y"].iloc[validation_positions], predictions)
        else:
            score = mean_squared_error(
                data["y"].iloc[validation_positions], predictions
            ) ** 0.5
        scores.append(score)

    return np.array(scores), np.mean(removed)


removal_methods = [
    "No removal", "IQR (3+ columns)", "Isolation Forest", "LOF",
    "KNN distance", "K-Means distance", "DBSCAN", "SOM distance",
]
removal_rows = []

for task, data in task_data.items():
    results = {
        method: evaluate_removal(data, method) for method in removal_methods
    }
    baseline = results["No removal"][0]

    for method, (scores, removed_percent) in results.items():
        comparison = paired_comparison(
            scores, baseline, task == "classification"
        )
        removal_rows.append({
            "task": task,
            "method": method,
            "mean_score": scores.mean(),
            "fold_std": scores.std(ddof=1),
            "removed_%": removed_percent,
            "mean_advantage": comparison["mean_advantage"],
            "standard_error": comparison["se"],
            "folds_better": comparison["folds_a_better"],
            "reliably_better": (
                comparison["mean_advantage"] > 2 * comparison["se"]
            ),
        })

removal_table = pd.DataFrame(removal_rows)
display(removal_table.round(4))

## 10. DBSCAN sensitivity

DBSCAN needs an `eps` choice. The table shows how several reasonable choices change the result on the same development data. A stable natural cluster structure would not change this sharply.

In [ ]:
matrix = scaled_numeric_matrix(X_dev, numeric_columns)
neighbour_distances = (
    NearestNeighbors(n_neighbors=10).fit(matrix).kneighbors(matrix)[0][:, -1]
)
dbscan_rows = []

for quantile in [0.90, 0.95, 0.97, 0.99]:
    epsilon = np.quantile(neighbour_distances, quantile)
    labels = DBSCAN(eps=epsilon, min_samples=10).fit_predict(matrix)
    cluster_count = len(set(labels)) - int(-1 in labels)
    dbscan_rows.append({
        "k-distance quantile": quantile,
        "eps": epsilon,
        "noise_%": np.mean(labels == -1) * 100,
        "clusters": cluster_count,
    })

display(pd.DataFrame(dbscan_rows).round(3))

## 11. Method-by-method judgement

| Method | Judgement for this dataset |
|---|---|
| Logical consistency rules | **Use.** They identify impossible values from feature definitions, including `prior_max_purchase_value > prior_total_spend`. |
| IQR | Keep as a univariate diagnostic only. It fails on zero-heavy columns and removing rows flagged in 3+ columns seriously harms regression. |
| `log1p` | Plausible for skewed monetary values, but it does not reliably improve either task in temporal validation. |
| Winsorisation | Small classification changes and no reliable improvement; it also discards real magnitude information. Do not use. |
| Robust scaling | Useful in some settings, but it is not reliably better than the task-specific scalers already chosen. |
| Isolation Forest | The most reasonable optional multivariate **diagnostic**, but its flagged rows are often valuable high-repeat/high-spend customers and removal is not reliably better. |
| LOF / KNN distance | Sensitive to local density and the high-dimensional distance definition; no reliable validation gain. |
| DBSCAN | Finds one broad cluster plus a parameter-sensitive noise set; reasonable `eps` choices change the flagged share substantially. |
| K-Means distance | K-Means is a clustering method, not a natural outlier definition. The result depends on an arbitrary number of clusters and does not reliably improve prediction. |
| SOM distance | Useful for visual exploration, but map size, training and distance threshold are arbitrary; removal does not reliably improve prediction. |

## Final decision

1. **Correct genuine data errors with domain and consistency rules.** The new maximum-versus-total rule affects 45 train values and 6 test values. They become missing and are imputed inside the model pipeline.
2. **Keep statistically extreme but plausible observations.** They represent real high-value purchases and customers. No row-removal method gives a reliable temporal-validation improvement; IQR removal is particularly damaging for regression.
3. **Do not add `log1p`, winsorisation or a different scaler for outliers.** Their small changes are within fold uncertainty. The existing classification and regression recipes remain simpler and equally reliable.
4. **Use IQR and Isolation Forest only for investigation if new data arrives.** They may highlight rows for manual review, but they are not automatic deletion rules.

Therefore the final pipeline uses **rule-based correction plus no statistical outlier deletion or capping**. The locked period and `test.csv` remain untouched.